# Phase 2 RV Feature Inspection

This notebook is inspection-only. It loads the frozen realized-variance outputs for US and India, checks the Phase 2 panel structure, and previews the generated diagnostics.

Production feature construction stays in `src/vrp/` and `scripts/build_features.py`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
REPORT_FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'

US_RV_PATH = DATA_PROCESSED_DIR / 'us_rv.parquet'
INDIA_RV_PATH = DATA_PROCESSED_DIR / 'india_rv.parquet'
RV_SUMMARY_PATH = REPORT_TABLE_DIR / 'rv_summary.csv'
RV_CORRELATIONS_PATH = REPORT_TABLE_DIR / 'rv_estimator_correlations.csv'
RV_METADATA_PATH = REPORT_TABLE_DIR / 'rv_metadata.json'


In [ ]:
us_rv = pd.read_parquet(US_RV_PATH)
india_rv = pd.read_parquet(INDIA_RV_PATH)

us_rv.head()

In [ ]:
{
    'us_columns': us_rv.columns.tolist(),
    'india_columns': india_rv.columns.tolist(),
    'us_rows': len(us_rv),
    'india_rows': len(india_rv),
}

In [ ]:
inspection = pd.DataFrame(
    {
        'market': ['US', 'INDIA'],
        'rv_gk_22d_ann_first_valid_index': [
            us_rv['rv_gk_22d_ann'].first_valid_index(),
            india_rv['rv_gk_22d_ann'].first_valid_index(),
        ],
        'rv_yz_22d_ann_first_valid_index': [
            us_rv['rv_yz_22d_ann'].first_valid_index(),
            india_rv['rv_yz_22d_ann'].first_valid_index(),
        ],
        'rv_cc_daily_first_value_is_nan': [
            pd.isna(us_rv.loc[0, 'rv_cc_daily']),
            pd.isna(india_rv.loc[0, 'rv_cc_daily']),
        ],
    }
)

inspection

In [ ]:
us_rv[[
    'date',
    'rv_cc_daily',
    'rv_parkinson_daily',
    'rv_gk_daily',
    'rv_rs_daily',
    'rv_gk_22d_ann',
    'rv_yz_22d_ann',
]].head(30)

# Repeat the same preview for India to compare the panels side by side.
india_rv[[
    'date',
    'rv_cc_daily',
    'rv_parkinson_daily',
    'rv_gk_daily',
    'rv_rs_daily',
    'rv_gk_22d_ann',
    'rv_yz_22d_ann',
]].head(30)

In [ ]:
missing_report = pd.DataFrame(
    {
        'market': ['US', 'INDIA'],
        'rv_gk_22d_ann_missing': [
            us_rv['rv_gk_22d_ann'].isna().sum(),
            india_rv['rv_gk_22d_ann'].isna().sum(),
        ],
        'rv_yz_22d_ann_missing': [
            us_rv['rv_yz_22d_ann'].isna().sum(),
            india_rv['rv_yz_22d_ann'].isna().sum(),
        ],
    }
)

summary = pd.read_csv(RV_SUMMARY_PATH)
correlations = pd.read_csv(RV_CORRELATIONS_PATH)
metadata = pd.read_json(RV_METADATA_PATH, typ='series')

{
    'missing_report': missing_report,
    'summary_markets': summary['market'].unique().tolist(),
    'correlation_rows': len(correlations),
    'metadata_phase': metadata.get('phase'),
}
